In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
import statsmodels.formula.api as smf

In [2]:
df = pd.read_csv("data__movies.csv", sep=",")

In [3]:
df= df.drop(columns=["homepage", "status", "tagline", "title","original_title", "overview", "id"])

In [4]:
df['production_companies'] = df['production_companies'].str.split(',').str[0]
df = df.map(lambda x: x.lower() if isinstance(x, str) else x)

In [5]:
df["genres"] = df["genres"].fillna("desconocido")
df["production_companies"] = df["production_companies"].fillna("otros")
df = df.dropna(subset=["release_date", "runtime"])

In [6]:
df["release_date"] = pd.to_datetime(df["release_date"], errors="coerce")
df = df.dropna(subset=["release_date"])
df["anio"] = df["release_date"].dt.year.astype(int)
df["mes"] = df["release_date"].dt.month.astype(int)
df = df.drop(columns=["release_date"])

In [7]:
df = df[(df["budget"] > 0) & (df["revenue"] > 0)
              & (df["runtime"] > 0) & (df["vote_count"] > 0)]


In [8]:
generos_exp = df["genres"].str.split(",").apply(
    lambda l: [g.strip() for g in l] if isinstance(l, list) else [])

In [9]:
top_generos = pd.Series([g for sub in generos_exp for g in sub]) \
                .value_counts().head(10).index.tolist()

In [10]:
def _limpiar(s):
    out = s
    for ch in [" ", ".", ",", "-", "(", ")", "/", "'", "&"]:
        out = out.replace(ch, "_")
    while "__" in out:
        out = out.replace("__", "_")
    return out.strip("_")

In [11]:
for g in top_generos:
    df[f"gen_{_limpiar(g)}"] = generos_exp.apply(lambda lst: int(g in lst))
df["n_generos"] = generos_exp.apply(len)

In [12]:
prods_exp = df["production_companies"].str.split(",").apply(
    lambda l: [p.strip() for p in l] if isinstance(l, list) else [])
top_prods = pd.Series([p for sub in prods_exp for p in sub]) \
              .value_counts().head(10).index.tolist()
for p in top_prods:
    df[f"prod_{_limpiar(p)}"] = prods_exp.apply(lambda lst: int(p in lst))

top_lang = df["original_language"].value_counts().head(5).index
df["original_language"] = df["original_language"].where(
    df["original_language"].isin(top_lang), "other")

In [13]:
df["budget"]     = np.log(df["budget"])
df["revenue"] = np.log(df["revenue"])
df["popularity"] = np.log1p(df["popularity"])
df["vote_count"] = np.log1p(df["vote_count"])

In [14]:
q_low  = df["revenue"].quantile(0.005)
q_high = df["revenue"].quantile(0.995)
df = df[(df["revenue"] >= q_low) & (df["revenue"] <= q_high)].copy()

for col in ["budget", "popularity", "vote_count",
            "vote_average", "runtime", "anio"]:
    df[col] = df[col] - df[col].mean()

In [15]:
df = df.reset_index(drop=True)

In [16]:
print(f"df_v2 final: {len(df):,} filas, {df.shape[1]} columnas")
print(f"Columnas creadas para géneros: {[c for c in df.columns if c.startswith('gen_')]}")
print(f"Columnas creadas para productoras: {[c for c in df.columns if c.startswith('prod_')]}")
df.head()

df_v2 final: 3,193 filas, 32 columnas
Columnas creadas para géneros: ['gen_drama', 'gen_comedy', 'gen_thriller', 'gen_action', 'gen_adventure', 'gen_romance', 'gen_crime', 'gen_science_fiction', 'gen_family', 'gen_fantasy']
Columnas creadas para productoras: ['prod_paramount_pictures', 'prod_universal_pictures', 'prod_columbia_pictures', 'prod_twentieth_century_fox_film_corporation', 'prod_new_line_cinema', 'prod_walt_disney_pictures', 'prod_village_roadshow_pictures', 'prod_miramax_films', 'prod_united_artists', 'prod_columbia_pictures_corporation']


,budget,genres,original_language,popularity,production_companies,revenue,runtime,vote_average,vote_count,anio,...,prod_paramount_pictures,prod_universal_pictures,prod_columbia_pictures,prod_twentieth_century_fox_film_corporation,prod_new_line_cinema,prod_walt_disney_pictures,prod_village_roadshow_pictures,prod_miramax_films,prod_united_artists,prod_columbia_pictures_corporation
0,2.684090,"adventure, fantasy, action",en,1.945155,walt disney pictures,20.683485,58.390229,0.589978,2.375647,5.349201,...,0,0,0,0,0,1,0,0,0,0
1,2.481565,"action, adventure, crime",en,1.688537,columbia pictures,20.596199,37.390229,-0.010022,2.368065,13.349201,...,0,0,1,0,0,0,0,0,0,0
2,2.540989,"action, adventure, science fiction",en,0.807962,walt disney pictures,19.464974,21.390229,-0.210022,1.625120,10.349201,...,0,0,0,0,0,1,0,0,0,0
3,2.533267,"fantasy, action, adventure",en,1.762528,columbia pictures,20.607711,28.390229,-0.410022,2.145872,5.349201,...,0,0,1,0,0,0,0,0,0,0
4,2.540989,"animation, family",en,0.908565,walt disney pictures,20.198671,-10.609771,1.089978,2.074620,8.349201,...,0,0,0,0,0,1,0,0,0,0


In [17]:
formula_final = '''
revenue ~ vote_count + budget + anio
+ gen_family + gen_science_fiction + gen_crime + gen_fantasy + gen_romance + gen_drama
+ vote_average
+ prod_new_line_cinema + prod_twentieth_century_fox_film_corporation
+ prod_paramount_pictures + prod_universal_pictures + prod_columbia_pictures
+ prod_miramax_films + prod_village_roadshow_pictures
+ runtime
+ budget:gen_crime
+ budget:gen_science_fiction
+ budget:gen_romance
+ budget:gen_fantasy
+ budget:gen_thriller
+ budget:vote_average
+ budget:runtime
+ budget:prod_twentieth_century_fox_film_corporation
+ budget:prod_new_line_cinema
+ vote_average:popularity
+ popularity:vote_count
'''

modelo_final = smf.ols(formula_final, data=df).fit(cov_type="HC3")
print(modelo_final.summary())

                            OLS Regression Results                            
Dep. Variable:                revenue   R-squared:                       0.664
Model:                            OLS   Adj. R-squared:                  0.661
Method:                 Least Squares   F-statistic:                     191.8
Date:                Sun, 26 Apr 2026   Prob (F-statistic):               0.00
Time:                        15:41:08   Log-Likelihood:                -4652.1
No. Observations:                3193   AIC:                             9364.
Df Residuals:                    3163   BIC:                             9546.
Df Model:                          29                                         
Covariance Type:                  HC3                                         
                                                         coef    std err          z      P>|z|      [0.025      0.975]
-------------------------------------------------------------------------------------------

In [22]:
# 1. Obtenemos las métricas de influencia y los residuos estudentizados del modelo actual
influencia = modelo_final.get_influence()
residuos_estudentizados = influencia.resid_studentized_internal

# 2. Filtramos los valores atípicos severos (errores estándar absolutos mayores a 2.0 o 2.5)
# Un umbral de 2.0 es más estricto y garantiza máxima normalidad.
df_limpio = df[abs(residuos_estudentizados) <= 2.0].copy()

# 3. Volvemos a ajustar el modelo con el dataframe limpio
modelo_corregido = smf.ols(formula_final, data=df_limpio).fit()

# 4. Mostramos los resultados
print(f"Observaciones eliminadas: {len(df) - len(df_limpio)}")
print(modelo_corregido.summary())

Observaciones eliminadas: 157
                            OLS Regression Results                            
Dep. Variable:                revenue   R-squared:                       0.766
Model:                            OLS   Adj. R-squared:                  0.764
Method:                 Least Squares   F-statistic:                     339.7
Date:               dom, 26 abr. 2026   Prob (F-statistic):               0.00
Time:                        15:44:13   Log-Likelihood:                -3334.3
No. Observations:                3036   AIC:                             6729.
Df Residuals:                    3006   BIC:                             6909.
Df Model:                          29                                         
Covariance Type:            nonrobust                                         
                                                         coef    std err          t      P>|t|      [0.025      0.975]
-------------------------------------------------------------

In [19]:
df_limpio

,budget,genres,original_language,popularity,production_companies,revenue,runtime,vote_average,vote_count,anio,...,prod_paramount_pictures,prod_universal_pictures,prod_columbia_pictures,prod_twentieth_century_fox_film_corporation,prod_new_line_cinema,prod_walt_disney_pictures,prod_village_roadshow_pictures,prod_miramax_films,prod_united_artists,prod_columbia_pictures_corporation
0,2.684090,"adventure, fantasy, action",en,1.945155,walt disney pictures,20.683485,58.390229,0.589978,2.375647,5.349201,...,0,0,0,0,0,1,0,0,0,0
1,2.481565,"action, adventure, crime",en,1.688537,columbia pictures,20.596199,37.390229,-0.010022,2.368065,13.349201,...,0,0,1,0,0,0,0,0,0,0
2,2.540989,"action, adventure, science fiction",en,0.807962,walt disney pictures,19.464974,21.390229,-0.210022,1.625120,10.349201,...,0,0,0,0,0,1,0,0,0,0
3,2.533267,"fantasy, action, adventure",en,1.762528,columbia pictures,20.607711,28.390229,-0.410022,2.145872,5.349201,...,0,0,1,0,0,0,0,0,0,0
4,2.540989,"animation, family",en,0.908565,walt disney pictures,20.198671,-10.609771,1.089978,2.074620,8.349201,...,0,0,0,0,0,1,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3188,-6.631611,comedy,en,0.035404,miramax films,14.963272,-18.609771,1.089978,0.591634,-7.650799,...,0,0,0,0,0,0,0,1,0,0
3189,-7.442542,"horror, comedy, crime",en,-1.282623,dreamland productions,15.607270,-17.609771,-0.110022,-1.326877,-29.650799,...,0,0,0,0,0,0,0,0,0,0
3190,-6.931716,"crime, horror, mystery, thriller",ja,-2.804440,daiei studios,11.502875,0.390229,1.089978,-1.877524,-4.650799,...,0,0,0,0,0,0,0,0,0,0
3191,-7.981538,"science fiction, drama, thriller",en,0.193726,thinkfilm,12.959280,-33.609771,0.589978,0.454316,2.349201,...,0,0,0,0,0,0,0,0,0,0
